# EQL auf einer Plan-Datenbank (Roboteraktionen)

Zeigt für Abfragen auf aufgezeichneten Plänen (z. B. welche Aktionen mit welchem Arm ausgeführt wurden) die EQL-Query, das vom Übersetzer generierte SQL und das **Ergebnis** der Query.

Die Datenbank wird hier **nicht** erzeugt. Sie entsteht einmalig in `eql_plan_setup.ipynb` (oder `create_plan_database.py`). Dieses Notebook öffnet nur die fertige SQLite-Datei.

Dieses Notebook läuft bewusst getrennt von `eql_show.ipynb` mit eigenem Kernel: Die Plan-ORM-Klassen (`coraplex_orm`) und die Testklassen aus `eql_data_setup` registrieren Tabellen in derselben SQLAlchemy-Umgebung und lassen sich nicht in einem Kernel mischen.

## Setup

In [1]:
import os
from IPython.display import Markdown, display
from sqlalchemy.orm import sessionmaker

import coraplex.orm.ormatic_interface as coraplex_orm
from coraplex.datastructures.enums import Arms
from krrood.entity_query_language.factories import entity, variable, an, set_of
from krrood.ormatic.eql_interface import eql_to_sql
from krrood.ormatic.utils import create_engine

PLAN_DATABASE_PATH = "/home/jovyan/work/tutorials/plan_result_rich.sqlite"

if not os.path.exists(PLAN_DATABASE_PATH):
    raise FileNotFoundError(
        f"{PLAN_DATABASE_PATH} fehlt. Zuerst eql_plan_setup.ipynb ausführen."
    )

plan_session = sessionmaker(bind=create_engine(f"sqlite+pysqlite:///{PLAN_DATABASE_PATH}"))()

MAX_ZEILEN = 20  


def zeige_plan(titel, query, eql_text):
    """Zeigt EQL, generiertes SQL und das Ergebnis der Query."""
    uebersetzt = eql_to_sql(query, plan_session)
    display(Markdown(
        f"### {titel}\n\n"
        f"**EQL**\n```python\n{eql_text.strip()}\n```\n\n"
        f"**Vom Übersetzer generiertes SQL**\n```sql\n{str(uebersetzt.sql_query).strip()}\n```\n\n"
        f"**Ergebnis**"
    ))
    zeilen = list(uebersetzt.evaluate())
    print(f"{len(zeilen)} Ergebnis(se)")
    for zeile in zeilen[:MAX_ZEILEN]:
        print(" ", zeile)
    if len(zeilen) > MAX_ZEILEN:
        print(f"  ... ({len(zeilen) - MAX_ZEILEN} weitere)")

print("Bereit.")

Bereit.


### P1. Welcher Arm wurde benutzt?

**Frage:** Welche Arme wurden bei Transportaktionen verwendet?

Die kürzeste Form einer Abfrage: ein einzelnes Attribut.

In [2]:
action = variable(type_=coraplex_orm.TransportActionDAO, domain=[])
query = an(set_of(action.arm))

zeige_plan("Welcher Arm wurde benutzt?", query, "an(set_of(action.arm))")

### Welcher Arm wurde benutzt?

**EQL**
```python
an(set_of(action.arm))
```

**Vom Übersetzer generiertes SQL**
```sql
SELECT "TransportActionDAO".arm 
FROM "DesignatorDAO" JOIN "ActionDescriptionDAO" ON "ActionDescriptionDAO".database_id = "DesignatorDAO".database_id JOIN "TransportActionDAO" ON "TransportActionDAO".database_id = "ActionDescriptionDAO".database_id
```

**Ergebnis**

3 Ergebnis(se)
  {TransportActionDAO.arm: LEFT}
  {TransportActionDAO.arm: LEFT}
  {TransportActionDAO.arm: RIGHT}


### P2. Objektnamen pro Arm

**Frage:** Welches Objekt wurde mit welchem Arm transportiert?

`action.object_designator.name.name` folgt zwei Beziehungen, ohne dass ein Fremdschlüssel angegeben wird.

In [3]:
action = variable(type_=coraplex_orm.TransportActionDAO, domain=[])
query = an(set_of(action.arm, action.object_designator.name.name))

zeige_plan("Objektnamen pro Arm", query, "an(set_of(action.arm, action.object_designator.name.name))")

### Objektnamen pro Arm

**EQL**
```python
an(set_of(action.arm, action.object_designator.name.name))
```

**Vom Übersetzer generiertes SQL**
```sql
SELECT "TransportActionDAO".arm, "PrefixedNameDAO_1".name 
FROM "DesignatorDAO" JOIN "ActionDescriptionDAO" ON "ActionDescriptionDAO".database_id = "DesignatorDAO".database_id JOIN "TransportActionDAO" ON "TransportActionDAO".database_id = "ActionDescriptionDAO".database_id JOIN ("WorldEntityDAO" AS "WorldEntityDAO_1" JOIN "WorldEntityWithIDDAO" AS "WorldEntityWithIDDAO_1" ON "WorldEntityWithIDDAO_1".database_id = "WorldEntityDAO_1".database_id JOIN "WorldEntityWithSimulatorPropertiesDAO" AS "WorldEntityWithSimulatorPropertiesDAO_1" ON "WorldEntityWithSimulatorPropertiesDAO_1".database_id = "WorldEntityWithIDDAO_1".database_id JOIN "SemanticAnnotationDAO" AS "SemanticAnnotationDAO_1" ON "SemanticAnnotationDAO_1".database_id = "WorldEntityWithSimulatorPropertiesDAO_1".database_id JOIN "HasRootKinematicStructureEntityDAO" AS "HasRootKinematicStructureEntityDAO_1" ON "HasRootKinematicStructureEntityDAO_1".database_id = "SemanticAnnotationDAO_1".database_id JOIN "HasRootBodyDAO" AS "HasRootBodyDAO_1" ON "HasRootBodyDAO_1".database_id = "HasRootKinematicStructureEntityDAO_1".database_id) ON "HasRootBodyDAO_1".database_id = "TransportActionDAO"._object_designator_id JOIN "PrefixedNameDAO" AS "PrefixedNameDAO_1" ON "PrefixedNameDAO_1".database_id = "WorldEntityDAO_1"._name_id
```

**Ergebnis**

3 Ergebnis(se)
  {TransportActionDAO.arm: LEFT, TransportActionDAO.object_designator.name.name: 'Bowl'}
  {TransportActionDAO.arm: LEFT, TransportActionDAO.object_designator.name.name: 'Spoon'}
  {TransportActionDAO.arm: RIGHT, TransportActionDAO.object_designator.name.name: 'Milk'}


### P3. Status der Aktionen

**Frage:** Welche Status haben die ausgeführten Aktionsknoten (`ActionNodeDAO`)?

In [4]:
node = variable(type_=coraplex_orm.ActionNodeDAO, domain=[])
query = an(set_of(node.status))

zeige_plan("Status der Aktionen", query, "an(set_of(node.status))")

### Status der Aktionen

**EQL**
```python
an(set_of(node.status))
```

**Vom Übersetzer generiertes SQL**
```sql
SELECT "PlanNodeDAO".status 
FROM "PlanEntityDAO" JOIN "PlanNodeDAO" ON "PlanNodeDAO".database_id = "PlanEntityDAO".database_id JOIN "DesignatorNodeDAO" ON "DesignatorNodeDAO".database_id = "PlanNodeDAO".database_id JOIN "ActionNodeDAO" ON "ActionNodeDAO".database_id = "DesignatorNodeDAO".database_id
```

**Ergebnis**

35 Ergebnis(se)
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>}
  {ActionNodeDAO.status: <LifeCycleValue

### P4. Alle Aktionen mit dem linken Arm

**Frage:** Welche Transportaktionen wurden mit dem linken Arm ausgeführt?

Die Bedingung liest sich wie in Python: `action.arm == Arms.LEFT`. Zurückgegeben werden vollständige Aktionen.

In [5]:
action = variable(type_=coraplex_orm.TransportActionDAO, domain=[])
query = an(entity(action).where(action.arm == Arms.LEFT))

zeige_plan("Alle Aktionen mit dem linken Arm", query, "an(entity(action).where(action.arm == Arms.LEFT))")

### Alle Aktionen mit dem linken Arm

**EQL**
```python
an(entity(action).where(action.arm == Arms.LEFT))
```

**Vom Übersetzer generiertes SQL**
```sql
SELECT "TransportActionDAO".database_id, "ActionDescriptionDAO".database_id AS database_id_1, "DesignatorDAO".database_id AS database_id_2, "DesignatorDAO".polymorphic_type, "TransportActionDAO".arm, "TransportActionDAO"._object_designator_id, "TransportActionDAO"._target_location_id, "TransportActionDAO"._grasp_description_id 
FROM "DesignatorDAO" JOIN "ActionDescriptionDAO" ON "ActionDescriptionDAO".database_id = "DesignatorDAO".database_id JOIN "TransportActionDAO" ON "TransportActionDAO".database_id = "ActionDescriptionDAO".database_id 
WHERE "TransportActionDAO".arm = :arm_1
```

**Ergebnis**

2 Ergebnis(se)
  TransportActionDAO(arm=LEFT, object_designator=BowlDAO(id=UUID('8bc7b74a-031b-4b55-9d22-d0deb47169f6'), class_label=None, objects=[], root=BodyDAO(id=UUID('d7a7fa20-42fa-43ad-aaa1-9a77268ab611'), visual=ShapeCollectionDAO(shapes=[<semantic_digital_twin.orm.ormatic_interface.ShapeCollectionDAO_shapes_association object at 0x76ed482cd070>], reference_frame=BodyDAO(...)), collision=ShapeCollectionDAO(shapes=[<semantic_digital_twin.orm.ormatic_interface.ShapeCollectionDAO_shapes_association object at 0x76ed482ff530>], reference_frame=BodyDAO(...)), inertial=InertialDAO(mass=1.0, center_of_mass=Point3MappingDAO(x=0.0, y=0.0, z=0.0), inertia=InertiaTensorDAO()), simulator_additional_properties=[], name=PrefixedNameDAO(name='bowl.stl', prefix=None)), simulator_additional_properties=[], name=PrefixedNameDAO(name='Bowl', prefix=None)), target_location=PoseMappingDAO(position=Point3MappingDAO(x=5.0, y=3.3, z=0.75, reference_frame=BodyDAO(id=UUID('53d10978-a64e-4720-843d-37e3d508

### P5. Welche Aktionen wurden ausgeführt?

**Frage:** Welche Status und welche Art von Aktion (`polymorphic_type`) gibt es?

Die Aktionsart steht in der Datenbank an einer Vererbungstabelle, in EQL erreicht man sie über `node.designator`.

In [6]:
node = variable(type_=coraplex_orm.ActionNodeDAO, domain=[])
query = an(set_of(node.status, node.designator.polymorphic_type))

zeige_plan("Welche Aktionen wurden ausgeführt?", query, "an(set_of(node.status, node.designator.polymorphic_type))")

### Welche Aktionen wurden ausgeführt?

**EQL**
```python
an(set_of(node.status, node.designator.polymorphic_type))
```

**Vom Übersetzer generiertes SQL**
```sql
SELECT "PlanNodeDAO".status, "DesignatorDAO_1".polymorphic_type 
FROM "PlanEntityDAO" JOIN "PlanNodeDAO" ON "PlanNodeDAO".database_id = "PlanEntityDAO".database_id JOIN "DesignatorNodeDAO" ON "DesignatorNodeDAO".database_id = "PlanNodeDAO".database_id JOIN "ActionNodeDAO" ON "ActionNodeDAO".database_id = "DesignatorNodeDAO".database_id JOIN "DesignatorDAO" AS "DesignatorDAO_1" ON "DesignatorDAO_1".database_id = "DesignatorNodeDAO"._designator_id
```

**Ergebnis**

35 Ergebnis(se)
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>, ActionNodeDAO.designator.polymorphic_type: 'ParkArmsActionDAO'}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>, ActionNodeDAO.designator.polymorphic_type: 'MoveTorsoActionDAO'}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>, ActionNodeDAO.designator.polymorphic_type: 'TransportActionDAO'}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>, ActionNodeDAO.designator.polymorphic_type: 'TransportActionDAO'}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>, ActionNodeDAO.designator.polymorphic_type: 'TransportActionDAO'}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>, ActionNodeDAO.designator.polymorphic_type: 'ParkArmsActionDAO'}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>, ActionNodeDAO.designator.polymorphic_type: 'ParkArmsActionDAO'}
  {ActionNodeDAO.status: <LifeCycleValues.NOT_STARTED: 0>, ActionNodeDAO.designator.polymorphic_type: 'MoveTorsoActionD